# SAM2 Baseline (Release)\n
\n
This notebook provides a clean SAM2 baseline for binary 2D segmentation.\n
It mirrors the one-fold finetuning idea from the original notebooks and uses a small training run for display.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from dataset import SAM2SliceDataset, build_manifest_from_metadata, one_fold_split
from train_eval import build_sam2_predictor, fit_sam2_one_fold

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [7]:
# Paths from original SAM2 workflow
metadata_csv = Path('./release_assets/external/DeePiCt/2d_cnn/metas/nucleus_mitochondria_metadata.csv')
sam2_repo = Path('./release_assets/external/SAM2')
checkpoint = sam2_repo / 'checkpoints' / 'sam2.1_hiera_small.pt'
model_cfg = sam2_repo / 'checkpoints' / 'small.yaml'

# Prefer the original prepared real-data manifest (same source as original notebooks).
orig_manifest = Path('/virtual_data/train_data/training/compare/SAM2/sam2_finetune/manifest.csv')

# Release outputs
out_dir = Path('./artifacts') / 'sam2_release'
data_dir = out_dir / 'finetune_data'
out_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

if orig_manifest.exists():
    manifest_csv = orig_manifest
    print('Using original real-data manifest:', manifest_csv)
else:
    manifest_csv = data_dir / 'manifest.csv'
    if not manifest_csv.exists():
        manifest_csv = build_manifest_from_metadata(
            metadata_csv=metadata_csv,
            out_root=data_dir,
            image_col='data',
            mask_col='labels',
            max_volumes=8,
            max_slices_per_volume=96,
        )
    print('Using release-generated manifest:', manifest_csv)

Using original real-data manifest: /virtual_data/train_data/training/compare/SAM2/sam2_finetune/manifest.csv


In [8]:
manifest_df = pd.read_csv(manifest_csv)
print('Total slices:', len(manifest_df))
print('Tomograms:', manifest_df['id'].nunique())

train_df, val_df = one_fold_split(manifest_df, train_tomos=5)
print('Train slices:', len(train_df), 'Val slices:', len(val_df))
print('Train tomograms:', sorted(train_df['id'].unique().tolist()))
print('Val tomogram:', sorted(val_df['id'].unique().tolist()))

Total slices: 1386
Tomograms: 6
Train slices: 1085 Val slices: 301
Train tomograms: ['TS_005', 'TS_023', 'TS_031', 'sllt0039', 'sllt0051']
Val tomogram: ['sllt0081']


In [9]:
# Small sampled split for quick display training
train_sample = train_df.sample(n=min(192, len(train_df)), random_state=SEED).reset_index(drop=True)
val_sample = val_df.sample(n=min(96, len(val_df)), random_state=SEED).reset_index(drop=True)

train_ds = SAM2SliceDataset(train_sample.to_dict('records'), class_id=1, augment=True)
val_ds = SAM2SliceDataset(val_sample.to_dict('records'), class_id=1, augment=False)

batch_size = 2
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

Train batches: 96 Val batches: 48


In [15]:
predictor = build_sam2_predictor(
    sam2_repo=sam2_repo,
    model_cfg=model_cfg,
    checkpoint=checkpoint,
    device=device,
 )

history = fit_sam2_one_fold(
    predictor=predictor,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=5,
    lr=1e-5,
    out_dir=out_dir,
    # max_train_steps=20,
    # max_val_batches=10,
 )

history_df = pd.DataFrame(history)
history_df.to_csv(out_dir / 'sam2_small_onefold_history.csv', index=False)
history_df

Epoch 01 | train_loss=0.2919 | val_iou=0.1186


Epoch 02 | train_loss=0.2519 | val_iou=0.1868


Epoch 03 | train_loss=0.2249 | val_iou=0.2577


Epoch 04 | train_loss=0.2126 | val_iou=0.3239


Epoch 05 | train_loss=0.1999 | val_iou=0.3286


,epoch,train_loss,val_iou
0,1,0.291932,0.118574
1,2,0.251861,0.186765
2,3,0.224866,0.257707
3,4,0.212556,0.323851
4,5,0.199878,0.328586
